### MFR Demo

In [1]:
import pandas as pd
from datasets import load_dataset
from utils import counting, mfr, evaluate

# Load data
dev_pub_data = load_dataset("weerayut/multilexnorm2026-dev-pub")

# Select data
lang = "en"
train, val = dev_pub_data["train"], dev_pub_data["validation"]

# Smoke test baseline
counts = counting(train)
out = mfr(['bcause', 'u', 'r', 'funny'], counts)
print("Smoke test:", out)

## Inference
ds = pd.DataFrame(val)
ds['pred'] = ds['raw'].apply(lambda x: mfr(x, counts))

## Evaluate 
evaluate(ds['raw'].tolist(), ds['norm'].tolist(), ds['pred'].tolist())

Smoke test: ['because', 'u', 'rồi', 'funny']
Baseline acc.(LAI): 88.48
Accuracy:           92.97
ERR:                39.02


(0.8847954569019836, 0.9297478803927355, 0.3901966214345056)

### Submission demo

In [ ]:
# data, save_path = load_dataset("weerayut/multilexnorm2026-dev-pub"), "outputs/submission_dev"
data, save_path = load_dataset("weerayut/multilexnorm2026-full-pub"), "outputs/submission_full"

In [3]:
def prediction(train, test):
    train_df = train.to_pandas()
    test_df  = test.to_pandas()

    # Build a per-language dictionary
    count_langs = {} 
    for lang in train_df["lang"].unique():
        train_lang = train_df.loc[train_df["lang"] == lang]
        count_langs[lang] = counting(train_lang.to_dict(orient="records"))

    # Prediction per language
    test_df["pred"] = test_df.apply(
        lambda r: mfr(r["raw"], count_langs.get(r["lang"])),
        axis=1
    )
    return test_df

In [4]:
from datasets import concatenate_datasets

## predict and save dev
train = concatenate_datasets([data['train'], data['validation']])
out = prediction(train, data['test'])
out.to_json(f"{save_path}/predictions.json", orient="records")

In [5]:
out[['raw', 'pred', 'lang']].head(3)

,raw,pred,lang
0,"[Jeg, skaelver, .]","[Jeg, skaelver, .]",da
1,"[Jeg, stryger, en, taendstik, .]","[Jeg, stryger, en, taendstik, .]",da
2,"[Jeg, tager, den, vaek, igen, .]","[Jeg, tager, den, væk, igen, .]",da


### Zip files

In [6]:
from utils import zip_files_flat

zip_files_flat(save_path, f"{save_path}.zip")

Created zip file: outputs/submission_dev.zip
